<a href="https://colab.research.google.com/github/shekinahudegbue/COSC-4371/blob/main/P15_checkpoint_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q datasets scikit-learn pandas numpy scipy matplotlib

In [ ]:
import re, sys, time, platform, unicodedata, subprocess, os
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from datasets import load_dataset
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

SEED = 4371
FPR_TARGET = 0.05
NEAR_DUP = 0.9
VAL_FRACTION_FOLDS = 5
REVISIONS = {"safeguard": "a3a877d", "deepset": "4f61ecb", "gandalf": "04737b6"}

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__)
print("Hardware:", platform.processor() or platform.machine(), "| CPUs:", os.cpu_count())

Python 3.13.16 | pandas 2.2.3 | numpy 2.1.3 | scikit-learn 1.6.1
Hardware: x86_64 | CPUs: 2


In [ ]:
def hf(name, rev):
    try:
        return load_dataset(name, revision=rev)
    except Exception as e:
        print(f"WARNING: could not load {name} at revision {rev} ({e}).\n"
              f"Loading the default revision instead. For a pinned run, put the full commit hash in REVISIONS.")
        return load_dataset(name)

def load_raw():
    specs = [("safeguard", "xTRam1/safe-guard-prompt-injection"),
             ("deepset", "deepset/prompt-injections"),
             ("gandalf", "Lakera/gandalf_ignore_instructions")]
    parts, info = [], {}
    for short, hub in specs:
        d = hf(hub, REVISIONS[short])
        first = list(d.keys())[0]
        info[short] = {"rows": sum(len(d[s]) for s in d), "cols": len(d[first].column_names),
                       "label_missing": 0 if short == "gandalf" else
                                        int(sum(d[s].to_pandas()["label"].isna().sum() for s in d))}
        for s in d:
            df = d[s].to_pandas()
            if short == "gandalf":
                df["label"] = 1
            parts.append(pd.DataFrame({"dataset": short, "orig_split": s, "idx": range(len(df)),
                                       "text": df["text"], "label": df["label"].fillna(-1).astype(int)}))
    raw = pd.concat(parts, ignore_index=True)
    raw["text"] = raw["text"].fillna("").astype(str)
    raw["row_id"] = raw["dataset"] + ":" + raw["orig_split"] + ":" + raw["idx"].astype(str)
    return raw, info

raw, info = load_raw()
print(raw.groupby(["dataset", "orig_split"]).size())

README.md:   0%|          | 0.00/2.76k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  497kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-bc92128b9288a6d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/777 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/111 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/112 [00:00<?, ? examples/s]

dataset    orig_split
deepset    test           116
           train          546
gandalf    test           112
           train          777
           validation     111
safeguard  test          2060
           train         8236
dtype: int64


In [ ]:
ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff"), None)
def norm(t):
    t = unicodedata.normalize("NFKC", t).translate(ZW).lower()
    return re.sub(r"\s+", " ", t).strip()

raw["norm"] = raw["text"].map(norm)


sim_vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5))
sim_vec.fit(raw["norm"].unique())

rows = []
for ds, g in raw.groupby("dataset", sort=False):
    n, pos = len(g), int(g["label"].sum())
    both = g.groupby("text")["label"].nunique()
    rows.append({"dataset": ds, "rows": n, "columns": info[ds]["cols"],
                 "benign": n - pos, "benign %": round(100 * (n - pos) / n, 1),
                 "injection": pos, "injection %": round(100 * pos / n, 1),
                 "missing label": info[ds]["label_missing"],
                 "empty text": int((g["text"].str.strip() == "").sum()),
                 "exact dup (text)": int(g.duplicated("text").sum()),
                 "exact dup (text+label)": int(g.duplicated(["text", "label"]).sum()),
                 "dup after normalization": int(g.duplicated("norm").sum()),
                 "texts with both labels": int((both > 1).sum())})
audit = pd.DataFrame(rows).set_index("dataset")
print(audit.T.to_string())


g = raw[raw.dataset == "deepset"]
X = sim_vec.transform(g["norm"])
S = (X @ X.T).toarray()
iu = np.triu_indices_from(S, 1)
same_norm = g["norm"].values[iu[0]] == g["norm"].values[iu[1]]
near_pairs = int(((S[iu] >= NEAR_DUP) & ~same_norm).sum())
print("\ndeepset near-duplicate pairs (cosine >= %.2f, excluding normalized-exact pairs): %d" % (NEAR_DUP, near_pairs))
print("Repeated entities: unavailable (no user/session/source column). Time range: unavailable (no timestamp).")

dataset                  safeguard  deepset  gandalf
rows                       10296.0    662.0   1000.0
columns                        2.0      2.0      2.0
benign                      7150.0    399.0      0.0
benign %                      69.4     60.3      0.0
injection                   3146.0    263.0   1000.0
injection %                   30.6     39.7    100.0
missing label                  0.0      0.0      0.0
empty text                     0.0      0.0      0.0
exact dup (text)             159.0      0.0      0.0
exact dup (text+label)       157.0      0.0      0.0
dup after normalization      161.0      0.0      1.0
texts with both labels         2.0      0.0      0.0

deepset near-duplicate pairs (cosine >= 0.90, excluding normalized-exact pairs): 22
Repeated entities: unavailable (no user/session/source column). Time range: unavailable (no timestamp).


In [ ]:
pool = raw[(raw.dataset == "deepset") & (raw.orig_split == "train")].copy()

lab = pool.groupby("norm")["label"].nunique()
conflict = set(lab[lab > 1].index)
n_conflict_removed = int(pool["norm"].isin(conflict).sum())
pool = pool[~pool["norm"].isin(conflict)].reset_index(drop=True)

Xp = sim_vec.transform(pool["norm"])
adj = sp.csr_matrix((Xp @ Xp.T).toarray() >= NEAR_DUP)
n_groups, groups = connected_components(adj, directed=False)

skf = StratifiedGroupKFold(n_splits=VAL_FRACTION_FOLDS, shuffle=True, random_state=SEED)
tr_i, va_i = next(skf.split(pool, pool["label"], groups))
pool["split"] = "train"
pool.loc[va_i, "split"] = "validation"

print("conflicting-label training rows set aside:", n_conflict_removed)
print("near-duplicate clusters in the training pool:", n_groups, "for", len(pool), "rows")
print(pool.groupby(["split", "label"]).size().unstack(fill_value=0))

conflicting-label training rows set aside: 0
near-duplicate clusters in the training pool: 530 for 546 rows
label         0    1
split               
train       279  158
validation   64   45


In [ ]:
test_defs = {"deepset-test": ("deepset", "test"),
             "safeguard-test": ("safeguard", "test"),
             "gandalf-test": ("gandalf", "test")}
Xtv = sim_vec.transform(pool["norm"])
tv_norm = set(pool["norm"])

tests, removal_rows = {}, []
for name, (ds, sp_) in test_defs.items():
    t = raw[(raw.dataset == ds) & (raw.orig_split == sp_)].copy()
    exact = t["norm"].isin(tv_norm).values
    sim = (sim_vec.transform(t["norm"]) @ Xtv.T).max(axis=1).toarray().ravel()
    near = (sim >= NEAR_DUP) & ~exact
    keep = ~(exact | near)
    removal_rows.append({"test_set": name, "rows before": len(t), "removed exact": int(exact.sum()),
                         "removed near-only": int(near.sum()), "rows kept": int(keep.sum()),
                         "kept positive": int(t.loc[keep, "label"].sum()),
                         "kept negative": int((1 - t.loc[keep, "label"]).sum())})
    tests[name] = t[keep].reset_index(drop=True)
print(pd.DataFrame(removal_rows).to_string(index=False))

      test_set  rows before  removed exact  removed near-only  rows kept  kept positive  kept negative
  deepset-test          116              0                  3        113             60             53
safeguard-test         2060              0                  2       2058            648           1410
  gandalf-test          112              0                  0        112            112              0


In [ ]:
tr = pool[pool.split == "train"]
va = pool[pool.split == "validation"]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
clf = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=SEED)

t0 = time.perf_counter()
Xtr = vec.fit_transform(tr["text"])
clf.fit(Xtr, tr["label"])
train_seconds = time.perf_counter() - t0
print(f"training time: {train_seconds:.2f} s")

s_val = clf.predict_proba(vec.transform(va["text"]))[:, 1]
y_val = va["label"].to_numpy()

def choose_threshold(y, s, fpr_target):
    benign = s[y == 0]
    for t in np.sort(np.unique(s)):
        if (benign >= t).mean() <= fpr_target:
            return float(t)
    return float(np.nextafter(s.max(), 2))
t_raw = choose_threshold(y_val, s_val, FPR_TARGET)
THRESHOLD = round(float(np.ceil(t_raw * 1e4)) / 1e4, 4)
val_fpr = (s_val[y_val == 0] >= THRESHOLD).mean()
val_rec = (s_val[y_val == 1] >= THRESHOLD).mean()
assert val_fpr <= FPR_TARGET
print(f"THRESHOLD = {THRESHOLD}  (validation FPR {val_fpr:.3f}, validation recall {val_rec:.3f}, "
      f"{int((y_val == 0).sum())} benign / {int((y_val == 1).sum())} injection validation rows)")

training time: 0.24 s
THRESHOLD = 0.4956  (validation FPR 0.031, validation recall 0.822, 64 benign / 45 injection validation rows)


In [ ]:
split_rows = [pool[["row_id", "dataset", "split", "label"]]]
score_rows = []
for name, t in tests.items():
    split_rows.append(t.assign(split="test")[["row_id", "dataset", "split", "label"]])
    s = clf.predict_proba(vec.transform(t["text"]))[:, 1]
    score_rows.append(pd.DataFrame({"row_id": t["row_id"], "test_set": name,
                                    "label": t["label"], "score": s}))
split_df = pd.concat(split_rows, ignore_index=True)
scores_df = pd.concat(score_rows, ignore_index=True)
assert not split_df["row_id"].duplicated().any()
split_df.to_csv("split.csv", index=False)
scores_df.to_csv("scores.csv", index=False)
print("wrote split.csv", split_df.shape, "and scores.csv", scores_df.shape)


all_test = pd.concat(tests.values())
print("\nTraining rows that exactly duplicate a test row (raw text):        ", int(tr["text"].isin(set(all_test["text"])).sum()))
print("Training rows that exactly duplicate a test row (normalized text): ", int(tr["norm"].isin(set(all_test["norm"])).sum()))
print("Training rows that exactly duplicate a validation row (raw text):  ", int(tr["text"].isin(set(va["text"])).sum()))
print("Training rows that exactly duplicate a validation row (normalized):", int(tr["norm"].isin(set(va["norm"])).sum()))

wrote split.csv (2829, 4) and scores.csv (2283, 4)

Training rows that exactly duplicate a test row (raw text):         0
Training rows that exactly duplicate a test row (normalized text):  0
Training rows that exactly duplicate a validation row (raw text):   0
Training rows that exactly duplicate a validation row (normalized): 0


In [ ]:
if os.path.exists("check_checkpoint.py"):
    out = subprocess.run([sys.executable, "check_checkpoint.py", "--split", "split.csv",
                          "--scores", "scores.csv", "--threshold", str(THRESHOLD)],
                         capture_output=True, text=True)
    print(out.stdout, out.stderr)
else:
    print("Upload check_checkpoint.py, then re-run this cell. Command:\n"
          f"python check_checkpoint.py --split split.csv --scores scores.csv --threshold {THRESHOLD}")

split.csv: 2829 rows. scores.csv: 2283 rows.

Rows per dataset and split
--------------------------
  dataset      split  rows  positive  negative  positive %
  deepset      train   437       158       279       36.16
  deepset validation   109        45        64       41.28
  deepset       test   113        60        53       53.10
  gandalf       test   112       112         0      100.00
safeguard       test  2058       648      1410       31.49

Metrics per test set (threshold 0.4956; positive when score >= threshold)
-------------------------------------------------------------------------
Each value is followed by its 95% bootstrap interval in brackets.

Test set: deepset-test
  test rows                 113
  positive rows             60 (53.10%)
  negative rows             53 (46.90%)
  bootstrap resamples       1000
  random-scorer AP          0.5310 (equals the positive share)
  random-scorer AUROC       0.5000
  precision                 1.0000 [1.0000, 1.0000]
  recall    